# Notebook 6 — Robustness sets for both datasets  ·  GPU: T4

200 fully correct answers per dataset (Mohler: gold ≥ 9, 40 per test fold; SciEntsBank: gold `correct`, ~67 per official test set), none containing negation words, each turned into five controlled variants:

| Variant | How | A good grader should… |
|---|---|---|
| `negated` | insert *not* after the first auxiliary/copula (else prefix "It is not the case that") | lower the score / move away from `correct` (ideally to `contradictory`) |
| `keyword_salad` | the reference answer's content words, shuffled, no syntax | lower the score / not `correct` |
| `swapped` | a correct answer to a *different* question | ≈0 / `irrelevant` or `non_domain` |
| `typos` | character errors in ~15% of words | keep the score / stay `correct` |
| `paraphrase` | reworded by Llama 3 8B Instruct (greedy), meaning preserved | keep the score / stay `correct` |

Scored by the **fine-tuned model** (Mohler: the answer's own fold model, so never seen in training; SciEntsBank: the test sets were never trained on), by **Instruct zero-shot** (SciEntsBank: calibrated, as in Notebook 2) and by the **lexical composite / lexical classifier**. Resumable; ≈40 min.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
ROOT = '/content/drive/MyDrive/asag_revision'
sys.path.insert(0, ROOT)
!nvidia-smi --query-gpu=name --format=csv

In [ ]:
!pip -q install -U bitsandbytes accelerate peft sentence-transformers language-tool-python

In [ ]:
import re, json, gc, time
import numpy as np, pandas as pd, torch
from tqdm.auto import tqdm
from asag_utils import *
from llm_utils import *
import nltk
for p in ['stopwords', 'wordnet', 'omw-1.4', 'punkt', 'punkt_tab']:
    nltk.download(p, quiet=True)
RES = f'{ROOT}/results'
mo = pd.read_csv(f'{ROOT}/data/mohler_prepared.csv')
se = pd.read_csv(f'{ROOT}/data/scientsbank_prepared.csv')
SET = {'mohler': f'{RES}/robustness_set.csv', 'scientsbank': f'{RES}/robustness_set_scientsbank.csv'}
def out(ds, what): return f'{RES}/robustness_scores_{what}.csv' if ds == 'mohler' else f'{RES}/robustness_scores_scientsbank_{what}.csv'

## 1. Build the rule-based variants (deterministic, seed 42)

In [ ]:
NEG_WORDS = re.compile(r"\b(not|no|never|none|nothing|cannot|without)\b|n't", re.I)
AUX = re.compile(r"\b(is|are|was|were|can|could|will|would|should|must|may|might|does|do|did|has|have|had)\b", re.I)

def negate(a):
    m = AUX.search(a)
    if m:
        return a[:m.end()] + ' not' + a[m.end():]
    return 'It is not the case that ' + a[0].lower() + a[1:]

def keyword_salad(ref, rng):
    toks = content_tokens(ref)
    return ' '.join(rng.permutation(toks)) if toks else ref

def typos(a, rng, rate=0.15):
    words = a.split()
    cand = [i for i, w in enumerate(words) if len(w) > 3 and w.isalpha()]
    if not cand:
        return a
    n = max(1, int(round(rate * len(words))))
    for i in rng.choice(cand, min(n, len(cand)), replace=False):
        w = words[i]; j = rng.integers(1, len(w) - 1); op = rng.integers(3)
        words[i] = (w[:j] + w[j + 1] + w[j] + w[j + 2:] if op == 0 else
                    w[:j] + w[j + 1:] if op == 1 else w[:j] + w[j] + w[j:])
    return ' '.join(words)

def build(picks, donors_by_part, part_col, rng, extra):
    rows = []
    for _, r in picks.iterrows():
        a = str(r.student_answer)
        donors = donors_by_part[r[part_col]]
        other = donors[donors.group != r.group].sample(1, random_state=int(rng.integers(10**9))).iloc[0]
        base = {'orig_uid': r.uid, part_col: r[part_col], 'group': r.group, 'question': r.question,
                'reference_answer': r.reference_answer, **{c: r[c] for c in extra}}
        for v, text in [('original', a), ('negated', negate(a)), ('keyword_salad', keyword_salad(r.reference_answer, rng)),
                        ('swapped', str(other.student_answer)), ('typos', typos(a, rng)), ('paraphrase', None)]:
            rows.append({**base, 'variant': v, 'student_answer': text})
    return pd.DataFrame(rows)

rng = np.random.default_rng(SEED)
sets = {}
if os.path.exists(SET['mohler']):
    sets['mohler'] = pd.read_csv(SET['mohler'])
else:
    picks, donors = [], {}
    for k in range(N_FOLDS):
        te = mo[mo[f'split_f{k}'] == 'test'].assign(fold=k)
        pool = te[(te.y >= 9) & ~te.student_answer.astype(str).str.contains(NEG_WORDS)]
        picks.append(pool.sample(40, random_state=SEED + k)); donors[k] = te[te.y >= 9]
    d = build(pd.concat(picks), donors, 'fold', rng, ['y'])
    d['rid'] = [f'r{i}' for i in range(len(d))]; sets['mohler'] = d
if os.path.exists(SET['scientsbank']):
    sets['scientsbank'] = pd.read_csv(SET['scientsbank'])
else:
    picks, donors = [], {}
    for t, n in [('test_ua', 67), ('test_uq', 67), ('test_ud', 66)]:
        te = se[se.split == t]
        pool = te[(te.label == 'correct') & ~te.student_answer.astype(str).str.contains(NEG_WORDS)]
        picks.append(pool.sample(n, random_state=SEED)); donors[t] = te[te.label == 'correct']
    d = build(pd.concat(picks), donors, 'split', rng, [])
    d['rid'] = [f's{i}' for i in range(len(d))]; sets['scientsbank'] = d
for ds, d in sets.items():
    print(ds, d.variant.value_counts().to_dict())
    for v in ['original', 'negated', 'keyword_salad', 'swapped', 'typos']:
        print(f'  {v:14s}', d[d.variant == v].student_answer.iloc[0][:140])

## 2. Instruct model: generate paraphrases, then score every variant zero-shot

In [ ]:
PARA = ('Paraphrase the following student answer using different wording but exactly the same meaning. '
        'Do not add or remove information. Reply with the paraphrase only.\n\nAnswer: {a}')
todo_para = any(d.student_answer.isna().any() for d in sets.values())
todo_zs = not all(os.path.exists(out(ds, 'instruct_zs')) for ds in sets)
if todo_para or todo_zs:
    model, tok = load_hf_4bit(INSTRUCT_MODEL)
    for ds, d in sets.items():
        need = d[d.student_answer.isna()]
        if len(need):
            origs = [d[(d.orig_uid == u) & (d.variant == 'original')].student_answer.iloc[0] for u in need.orig_uid]
            outs = []
            for i in tqdm(range(0, len(origs), 8), desc=f'paraphrase {ds}'):
                batch = [tok.apply_chat_template([{'role': 'user', 'content': PARA.format(a=a)}], tokenize=False,
                                                 add_generation_prompt=True) for a in origs[i:i + 8]]
                enc = tok(batch, return_tensors='pt', padding=True, add_special_tokens=False).to(model.device)
                with torch.no_grad():
                    gen = model.generate(**enc, max_new_tokens=160, do_sample=False, pad_token_id=tok.pad_token_id)
                outs += [g.strip().strip('"') for g in tok.batch_decode(gen[:, enc.input_ids.shape[1]:], skip_special_tokens=True)]
            d.loc[need.index, 'student_answer'] = outs
            for o, p in list(zip(origs, outs))[:3]:
                print('ORIG:', o[:140]); print('PARA:', p[:140]); print()
        d.to_csv(SET[ds], index=False)
        if not os.path.exists(out(ds, 'instruct_zs')):
            name = 'mohler' if ds == 'mohler' else 'scientsbank'
            words = SCORE_TARGETS if ds == 'mohler' else label_candidates(chat=True)
            cand, _ = candidate_ids(tok, words)
            texts = to_chat(tok, make_prompts(d, name), INSTRUCT_PREFILL[name])
            lp = candidate_logprobs(model, tok, texts, cand, batch_size=8, add_special_tokens=False)
            cols = [f'lp_{w.strip()}' for w in words] if ds == 'mohler' else [f'lp_{l}' for l in LABEL_ORDER]
            pd.concat([d[['rid']].reset_index(drop=True), pd.DataFrame(lp, columns=cols)], axis=1).to_csv(out(ds, 'instruct_zs'), index=False)
    del model; gc.collect(); torch.cuda.empty_cache()
else:
    print('paraphrases and zero-shot scores already done')

## 3. Fine-tuned models (base model loaded once; LoRA adapters swapped)

In [ ]:
if all(os.path.exists(out(ds, 'ft')) for ds in sets):
    print('fine-tuned scores already done')
else:
    from peft import PeftModel
    model, tok = load_hf_4bit(BASE_MODEL)
    model = PeftModel.from_pretrained(model, f'{ROOT}/models/mohler_llama3_ft_f0', adapter_name='f0')
    for k in range(1, N_FOLDS):
        model.load_adapter(f'{ROOT}/models/mohler_llama3_ft_f{k}', adapter_name=f'f{k}')
    model.load_adapter(f'{ROOT}/models/scientsbank_llama3_ft', adapter_name='se')
    model.eval()
    d = sets['mohler']; cand, _ = candidate_ids(tok, SCORE_TARGETS); res = []
    for k in range(N_FOLDS):
        model.set_adapter(f'f{k}'); dk = d[d.fold == k]
        lp = candidate_logprobs(model, tok, make_prompts(dk, 'mohler'), cand, batch_size=8)
        res.append(pd.concat([dk[['rid']].reset_index(drop=True), pd.DataFrame(lp, columns=[f'lp_{s}' for s in SCORE_TARGETS])], axis=1))
    pd.concat(res).to_csv(out('mohler', 'ft'), index=False)
    d = sets['scientsbank']; cand, _ = candidate_ids(tok, label_candidates(chat=False))
    model.set_adapter('se')
    lp = candidate_logprobs(model, tok, make_prompts(d, 'scientsbank'), cand, batch_size=8)
    pd.concat([d[['rid']].reset_index(drop=True), pd.DataFrame(lp, columns=[f'lp_{l}' for l in LABEL_ORDER])], axis=1).to_csv(out('scientsbank', 'ft'), index=False)
    del model; gc.collect(); torch.cuda.empty_cache()

## 4. Lexical systems (fitted exactly as in Notebook 0)

In [ ]:
if all(os.path.exists(out(ds, 'lexical')) for ds in sets):
    print('lexical scores already done')
else:
    import language_tool_python
    from sentence_transformers import SentenceTransformer
    from sklearn.linear_model import LinearRegression, LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import make_pipeline
    tool = language_tool_python.LanguageTool('en-US')
    st = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    feats = pd.read_csv(f'{ROOT}/cache/features.csv')
    COMP = ['context', 'keyword', 'synonym', 'grammar']
    for ds, d in sets.items():
        L = d[['rid', 'reference_answer', 'student_answer']].copy()
        e1 = st.encode(L.reference_answer.astype(str).tolist(), normalize_embeddings=True, batch_size=64)
        e2 = st.encode(L.student_answer.astype(str).tolist(), normalize_embeddings=True, batch_size=64)
        L['context'] = context_score((e1 * e2).sum(1))
        L['keyword'] = [keyword_score(r, s) for r, s in zip(L.reference_answer, L.student_answer)]
        L['synonym'] = [synonym_score(r, s) for r, s in zip(L.reference_answer, L.student_answer)]
        L['grammar'] = [grammar_score(len(tool.check(str(s)))) for s in tqdm(L.student_answer, desc=f'grammar {ds}')]
        L['composite_fixed'] = composite_fixed(L)
        if ds == 'mohler':           # original-weight composite, rescaled per fold (train+val)
            M = mo.merge(feats, on='uid'); L['score'] = np.nan
            for k in range(N_FOLDS):
                fit = M[M[f'split_f{k}'] != 'test']
                lr = LinearRegression().fit(fit[['composite_fixed']], fit.y)
                idx = (d.fold == k).values
                L.loc[idx, 'score'] = np.clip(lr.predict(L.loc[idx, ['composite_fixed']]), 0, 10)
        else:                        # 5-way logistic regression on the four components (train+val)
            S = se.merge(feats, on='uid'); fit = S[S.split.isin(['train', 'val'])]
            clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(fit[COMP], fit.label)
            L['label_pred'] = clf.predict(L[COMP])
        L.drop(columns=['reference_answer', 'student_answer']).to_csv(out(ds, 'lexical'), index=False)

## 5. Mohler results: change in score (0–10) relative to the original answer

In [ ]:
VARIANTS = ['negated', 'keyword_salad', 'swapped', 'typos', 'paraphrase']

def cluster_ci(values, groups, n_boot=2000, seed=SEED):
    values, groups = np.asarray(values, float), np.asarray(groups)
    sets_ = [np.flatnonzero(groups == g) for g in pd.unique(groups)]
    rng = np.random.default_rng(seed)
    b = [values[np.concatenate([sets_[i] for i in rng.integers(0, len(sets_), len(sets_))])].mean() for _ in range(n_boot)]
    return np.percentile(b, 2.5), np.percentile(b, 97.5)

d = sets['mohler'][['rid', 'orig_uid', 'group', 'variant']]
S = d.copy()
for name, what in [('llama3_ft', 'ft'), ('llama3_instruct_zs', 'instruct_zs')]:
    z = pd.read_csv(out('mohler', what))
    e, _, m = score_from_logprobs(z[[f'lp_{s}' for s in SCORE_TARGETS]].values)
    S = S.merge(pd.DataFrame({'rid': z.rid, name: e}), on='rid')
    print(name, 'mean valid mass', round(m.mean(), 3))
S = S.merge(pd.read_csv(out('mohler', 'lexical'))[['rid', 'score']].rename(columns={'score': 'lexical_composite'}), on='rid')
SYS_M = ['llama3_ft', 'llama3_instruct_zs', 'lexical_composite']
orig = S[S.variant == 'original'].set_index('orig_uid')
rows = []
for s in SYS_M:
    rows.append({'system': s, 'variant': 'original', 'mean_score': orig[s].mean()})
    for v in VARIANTS:
        dv = S[S.variant == v].set_index('orig_uid')
        delta = dv[s] - orig.loc[dv.index, s]
        lo, hi = cluster_ci(delta.values, dv.group.values)
        rows.append({'system': s, 'variant': v, 'mean_score': dv[s].mean(), 'mean_delta': delta.mean(),
                     'delta_ci': f'[{lo:.2f}, {hi:.2f}]', 'share_drop_ge3': (delta <= -3).mean(),
                     'share_abs_delta_le1': (delta.abs() <= 1).mean()})
rob_m = pd.DataFrame(rows); rob_m.to_csv(f'{RES}/robustness_summary.csv', index=False)
display(rob_m.round(3))

Good grader: large negative `mean_delta` and high `share_drop_ge3` for *negated*, *keyword_salad*, *swapped*; `mean_delta` ≈ 0 and high `share_abs_delta_le1` for *typos*, *paraphrase*.

## 6. SciEntsBank results: predicted-label distribution per variant

In [ ]:
from sklearn.linear_model import LogisticRegression
d = sets['scientsbank'][['rid', 'orig_uid', 'group', 'variant']]
P = d.copy()
lpc = [f'lp_{l}' for l in LABEL_ORDER]
z = pd.read_csv(out('scientsbank', 'ft'))
_, m, pr = labels_from_logprobs(z[lpc].values)
P = P.merge(pd.DataFrame({'rid': z.rid, 'llama3_ft': pr['5way']}), on='rid'); print('ft mean valid mass', round(m.mean(), 3))
# Instruct zero-shot, calibrated exactly as in Notebook 2 (logistic regression on train+val log-probs)
cal = pd.read_csv(f'{RES}/scientsbank_llama3_instruct_zs.csv').drop_duplicates('uid').merge(se[['uid', 'split', 'label']], on='uid')
cal = cal[cal.split.isin(['train', 'val'])]
qc, _, _ = labels_from_logprobs(cal[lpc].values)
clf = LogisticRegression(max_iter=2000).fit(np.log(np.clip(qc, 1e-12, 1)), cal.label)
z = pd.read_csv(out('scientsbank', 'instruct_zs'))
q, m, _ = labels_from_logprobs(z[lpc].values)
P = P.merge(pd.DataFrame({'rid': z.rid, 'llama3_instruct_zs_cal': clf.predict(np.log(np.clip(q, 1e-12, 1)))}), on='rid')
print('instruct mean valid mass', round(m.mean(), 3))
P = P.merge(pd.read_csv(out('scientsbank', 'lexical'))[['rid', 'label_pred']].rename(columns={'label_pred': 'lexical_4'}), on='rid')
SYS_S = ['llama3_ft', 'llama3_instruct_zs_cal', 'lexical_4']
TARGET = {'original': ('correct',), 'negated': ('contradictory',), 'keyword_salad': ('partially_correct_incomplete', 'contradictory', 'irrelevant', 'non_domain'),
          'swapped': ('irrelevant', 'non_domain'), 'typos': ('correct',), 'paraphrase': ('correct',)}
rows = []
for s in SYS_S:
    for v in ['original'] + VARIANTS:
        dv = P[P.variant == v]
        ok = dv[s].isin(TARGET[v]).astype(float)
        lo, hi = cluster_ci(ok.values, dv.group.values)
        rows.append({'system': s, 'variant': v, 'share_desired': ok.mean(), 'ci': f'[{lo:.2f}, {hi:.2f}]',
                     'desired': '/'.join(TARGET[v]), 'share_still_correct': (dv[s] == 'correct').mean(),
                     **{f'pred_{l}': (dv[s] == l).mean() for l in LABEL_ORDER}})
rob_s = pd.DataFrame(rows); rob_s.to_csv(f'{RES}/robustness_summary_scientsbank.csv', index=False)
display(rob_s.round(3))

`share_desired`: *original*, *typos*, *paraphrase* → still `correct`; *negated* → `contradictory`; *keyword_salad* → anything but `correct`; *swapped* → `irrelevant`/`non_domain`. `share_still_correct` is the key failure rate for the three meaning-destroying variants.

## 7. Figure and examples

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
w = 0.26
for i, s in enumerate(SYS_M):
    dd = rob_m[(rob_m.system == s) & (rob_m.variant != 'original')].set_index('variant').loc[VARIANTS]
    ax[0].bar(np.arange(len(VARIANTS)) + (i - 1) * w, dd.mean_delta, w, label=s)
ax[0].axhline(0, color='k', lw=.8); ax[0].set_xticks(range(len(VARIANTS))); ax[0].set_xticklabels(VARIANTS, fontsize=8)
ax[0].set_title('Mohler: mean change in score vs original'); ax[0].legend(fontsize=8)
for i, s in enumerate(SYS_S):
    dd = rob_s[rob_s.system == s].set_index('variant').loc[['original'] + VARIANTS]
    ax[1].bar(np.arange(6) + (i - 1) * w, dd.share_still_correct, w, label=s)
ax[1].set_xticks(range(6)); ax[1].set_xticklabels(['original'] + VARIANTS, fontsize=8)
ax[1].set_title("SciEntsBank: share still labelled 'correct'"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{RES}/fig_robustness.png', dpi=300); plt.show()

for ds, frame, systems in [('mohler', S, SYS_M), ('scientsbank', P, SYS_S)]:
    ex = sets[ds].merge(frame[['rid'] + systems], on='rid')
    ex = ex[ex.orig_uid == ex.orig_uid.iloc[0]][['variant', 'student_answer'] + systems]
    ex.to_csv(f'{RES}/robustness_example_{ds}.csv', index=False); display(ex.round(2))